# The download cache

`transitio` keeps every feed that `fetch` downloads in its **download
cache**, so that you can repeat an analysis with the same feeds, even
offline. The cache is the `gtfs` folder under `cache_dir`, which is by
default your platform's cache directory. It holds one version of a feed per
distinct archive, named by the archive's SHA-256 checksum.

The cache works as follows:

- A later call uses a cached version that serves its study day instead of
  downloading the feed again, also offline.
- A repeated request uses the version it used before, so a repeated run
  delivers the same feeds and reports.
- For a day that no cached version serves, `fetch` downloads the feed again
  and keeps the older versions.
- The crops, repairs and validation results made from a version are kept
  with it, and a call that makes the same ones reads them back. So are the
  trips found to repeat those of the other feeds the call delivers.

## Reuse a download

Let's fetch the feed recommended for Turku, which the
[Quickstart](quickstart.ipynb) already downloaded:

In [1]:
import transitio

rec = transitio.place("Turku").recommend("2026-10-14")
result = transitio.fetch(feeds=rec, osm=False)
result.selection_table()[["feed_id", "cache"]]

Fetching 1 feed for Turku


[1/1] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


Done in 2 s: 1 feed delivered, 0 skipped


,feed_id,cache
0,f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...,reused


Notice that the `cache` column says `reused`: the call delivered the cached
version instead of downloading the feed again.

## Download again

```python
transitio.fetch(feeds=rec, use_cache=False)
```

`use_cache=False` downloads every feed again and replaces its cached
versions. When every attempt for a feed fails, `fetch` delivers the version
a cached call would use, with a warning.

A call can decide differently when the cache already holds the feeds than
when it is empty, in three cases:

- A cached version that serves the day is used without the check that
  `expired="skip"` otherwise makes: a conditional request to the feed's URL
  that asks whether the archive has changed since the index saw it.
- For `contained="drop"`, a cached feed counts as unchanged since it was
  indexed only when that check proved it so under the index snapshot in use.
- For a feed that needs an account, any credentials for its provider count
  the same, whichever key they hold, and a call without credentials uses only
  copies fetched without them.

## Keep the delivered feeds in a folder

With `directory=`, `fetch` copies the delivered feeds into that folder, each
named by its feed id: `f-nvbw~ding.zip`, next to its provenance sidecar
`f-nvbw~ding.provenance.json`, which records where and when the feed was
downloaded. An id that is not a safe file name, such as
`f-u2f-pražskáintegrovanádoprava`, becomes its ASCII form followed by `+`
and the id's SHA-256 checksum: `f-u2f-prazskaintegrovanadoprava+<sha256>.zip`.

A later call that delivers the same feed into the same folder replaces its
files, and a call that leaves the feed out, because it repeats other feeds'
trips, removes its file. So give each call its own folder when calls run at
the same time, or when you want to keep their results side by side.

Without `directory=`, the delivered feeds are the files in the cache, which
are read-only on Linux and macOS. Either way, `result.paths` maps each
delivered feed's id to its file:

In [2]:
result = transitio.fetch(feeds=rec, osm=False, directory="feeds")
for feed_id, path in result.paths.items():
    print(feed_id, path)

Fetching 1 feed for Turku


[1/1] f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja: cached copy reused


Done in 2 s: 1 feed delivered, 0 skipped


f-u6x-turunlinja~autoilijainosakeyhtiö~savonlinjaoy~sl~autolinja feeds/f-u6x-turunlinjaautoilijainosakeyhtiosavonlinjaoyslautolinja+ca23bf3049e064ba45a72fc64e47bf328192c3b3765e1e2eca2356351af63b34.zip


## List and clear the cache

With `transitio.cache`, you can list what the cache holds and clear it.
`info()` gives one row per cached version, with its sizes:

In [3]:
info = transitio.cache.info()
info[["feed_id", "archive_bytes", "outputs_bytes", "served", "retrieved_at"]]

,feed_id,archive_bytes,outputs_bytes,served,retrieved_at
0,f-ud-tallinn,5679909,9917419,1,2026-10-09T13:58:16.096231+00:00
1,mdb-864,13115567,640,0,2026-10-09T14:01:10.206210+00:00
2,f-mdb-3215,529523655,29170627,1,2026-10-08T09:12:44.955173+00:00
3,f-u6x-turunlinja~autoilijainosakeyhtiö~savonli...,13135293,28898530,2,2026-10-09T18:50:45.811224+00:00
4,f-vr~fi,5610438,2361218,1,2026-10-09T13:58:06.625410+00:00
5,f-mdb-3235,48903755,53753429,1,2026-10-09T13:58:21.998541+00:00


`clear()` removes cached versions, either all of them or those unused for a
while:

```python
import datetime

transitio.cache.clear(older_than=datetime.timedelta(days=30))  # unused 30 days
transitio.cache.clear()       # everything
```

`clear` waits for a fetch that is working on a feed. A feed that a running
fetch adds after `clear` started stays in the cache until the next `clear`.